<a href="https://colab.research.google.com/github/mukailaalhshituabr-cyber/lab-4-llm-decision-support/blob/main/Lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Part 0: Repository and API-key setup
# API-key setup cell
import os

try:
    from google.colab import userdata
    API_KEY = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    API_KEY = os.environ["GROQ_API_KEY"]

from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


Section 1 — Talking to an LLM Programmatically

In [2]:
# Part 1.1 — Your first API call
# TODO: Write a helper function you will reuse for the WHOLE lab:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

question = "What is microfinance, in one sentence?"
print(ask_llm(question))

# Call the API directly (not through the helper) so we can inspect token usage
# TODO: Call it once with a simple question and print the answer.
raw_response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
)
# TODO: Print response.usage as well — how many tokens did your call consume?
print(raw_response.usage)

Microfinance refers to a type of financial service that provides small loans, savings, and other financial products to low-income individuals or small businesses, often in developing countries, to help them access capital and improve their economic well-being.
CompletionUsage(completion_tokens=37, prompt_tokens=44, total_tokens=81, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.055596684, prompt_time=0.001184741, completion_time=0.095370883, total_time=0.096555624)


system vs user: system sets persistent behaviour/rules for the whole conversation (e.g. "be factual and neutral"). user carries the specific request for this turn (the letter text, or a question).
What is a token: roughly a chunk of text, often about 3/4 of a word. Providers bill per token because token count drives the actual compute cost, every token is processed by the model, so per-token billing ties price to the work done rather than a flat fee per request.

In [4]:
# Part 1.2 — Temperature: the randomness dial

# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

question = "Suggest a name for a savings product for market traders in Accra."

low_temp_answers = [ask_llm(question, temperature=0.0) for _ in range(5)]
high_temp_answers = [ask_llm(question, temperature=1.2) for _ in range(5)]

# TODO: Print all 10 answers, grouped by temperature.
print("temperature = 0.0 ")
for i, a in enumerate(low_temp_answers, 1):
    print(f"{i}. {a}\n")

print("temperature = 1.2 ")
for i, a in enumerate(high_temp_answers, 1):
    print(f"{i}. {a}\n")

temperature = 0.0 
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect". This name could appeal to market traders who want to gather and save their earnings.
4. **Market Mobi**: This name incorporates "mobi", short for mobile, to suggest a convenient and accessible savings product.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language, implying a sense of community and mutual support among market traders.
6. **Kae Dwa**: "Kae Dwa" means "good fortune" or "prosperity" in the Ga language, which could be an attractive name for a savings product.
7. **Traders' Trust**: This name emphasizes the idea of trust and reliability, which is esse

At 0.0 the 5 answers are identical or nearly identical, the model deterministically picks its highest-probability words every time. At 1.2 the 5 answers vary noticeably in wording and even in the name suggested. For the loan decision-support system, low temperature (0) is the right choice for summarization, extraction, and briefs: the officer needs the same letter to produce the same facts every run, creativity here only adds inconsistency and hallucination risk, not value.